# 11 - Second reader results

The analysis fixed in notebook 10, section 5, on the answers of `scripts/run_pipeline.py`. Two LLM models read 203 accounts, 29 per group, each in six conditions. An answer is correct when its decision matches the account's label. Only counts and shares appear here; answers, reasons and account keys stay in `data/interim`.

Every group has the same size, so no share here describes the population. Intervals are 95% Wilson.

In [1]:
import json

import numpy as np
import pandas as pd
from scipy.stats import binomtest

from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.second_reader import (
    CONDITIONS,
    GROUP_NAMES,
    K1,
    K2,
    K3_D33,
    K3_D66,
    K3_D100,
    NOT_FLAGGED,
    NOT_SUSPICIOUS,
    SCORE_ONLY,
    SUSPICIOUS,
)

PIPELINE = DATA_INTERIM / "pipeline"
MODELS = ["qwen3.5:9b-q4_K_M", "gemma4:latest"]
DOSE_STEPS = [K2, K3_D33, K3_D66, K3_D100]
FLAGGED_GROUPS = GROUP_NAMES[:4]

results = pd.read_parquet(PIPELINE / "results.parquet")
reading = pd.read_parquet(PIPELINE / "reading.parquet", columns=["label", "group"])
label = results["account"].map(reading["label"])
has_output = results["shown_level"].notna()
shown_alert = has_output & (results["shown_level"] != NOT_FLAGGED)
says_suspicious = results["decision"] == SUSPICIOUS
results["truth"] = label.map({1: SUSPICIOUS, 0: NOT_SUSPICIOUS})
results["valid"] = results["decision"].notna() & results["confidence"].notna()
results["correct"] = results["valid"] & (results["decision"] == results["truth"])
results["shown_wrong"] = has_output & (shown_alert != (label == 1))
results["follows"] = results["valid"] & has_output & (says_suspicious == shown_alert)


def wilson(k: int, n: int, z: float = 1.959964) -> tuple[float, float]:
    """The 95% Wilson interval of k in n."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(centre - half, 0.0), min(centre + half, 1.0)


def rate(k: int, n: int) -> dict:
    low, high = wilson(k, n)
    return {"n": n, "k": k, "share": k / n if n else np.nan, "low": low, "high": high}


columns = ["valid", "correct", "shown_wrong", "follows", "decision"]
wide = results.set_index(["model", "account", "condition"])[columns].unstack(
    "condition"
)
group_of = results.drop_duplicates("account").set_index("account")["group"]
print(f"answers {len(results):,}  accounts {results['account'].nunique()}")

answers 2,436  accounts 203


## 1. The answers are those of the fixed plan

Both models answered every account in every condition, the accounts are the sample of notebook 10, thinking was off, and each model ran at the single version seen in the model check. The agreement with the shown output computed here must equal the one the pipeline stored.

In [2]:
cases = pd.read_parquet(DATA_INTERIM / "llm_experiment_cases.parquet")["group"]
assert sorted(results["model"].unique()) == sorted(MODELS)
for model in MODELS:
    mine = results[results["model"] == model]
    assert len(mine) == len(cases) * len(CONDITIONS) == 1218
    assert set(mine["account"]) == set(cases.index)
    per_account = mine.groupby("account")["condition"].agg(list)
    assert per_account.map(lambda c: sorted(c) == sorted(CONDITIONS)).all()
assert (results["account"].map(cases) == results["group"]).all()
assert (results["account"].map(reading["group"]) == results["group"]).all()
assert not results["think"].any()

lines = (DATA_INTERIM / "llm_trial.jsonl").read_text(encoding="utf-8").splitlines()
trial = pd.DataFrame([json.loads(line) for line in lines])
versions = results.groupby("model")["digest"].unique()
assert versions.map(len).eq(1).all()
for model in MODELS:
    assert versions[model][0] == trial.loc[trial["model"] == model, "digest"].iloc[0]

stored = results["valid"] & has_output
assert (results.loc[stored, "follows"] == results.loc[stored, "follows_model"]).all()
print("sample, conditions, versions and agreement as fixed: True")
print()
print(cases.value_counts().reindex(GROUP_NAMES).to_string())

sample, conditions, versions and agreement as fixed: True

group
strong, model right            29
strong, model wrong            29
uncertain, model right         29
uncertain, model wrong         29
missed, in the pattern file    29
missed, outside the file       29
not flagged, clean             29


## 2. Invalid answers

An answer is invalid when it has no decision or no confidence. Such an account leaves every comparison that uses that condition. The context column is the most tokens one call used, against a limit of 4,096.

In [3]:
invalid = (~results["valid"]).groupby([results["model"], results["condition"]]).sum()
print("invalid answers")
print(invalid.unstack().reindex(index=MODELS, columns=CONDITIONS).to_string())
print()
results["context"] = results["prompt_tokens"] + results["answer_tokens"]
calls = results.groupby("model").agg(
    median_seconds=("seconds", "median"),
    most_context=("context", "max"),
    from_the_cache=("cached", "sum"),
)
print(calls.reindex(MODELS).round(1).to_string())

invalid answers
condition          K1  K2  K3_d33  K3_d66  K3_d100  K3_score_only
model                                                            
qwen3.5:9b-q4_K_M   0   0       0       0        0              0
gemma4:latest       0   0       0       0        0              0

                   median_seconds  most_context  from_the_cache
model                                                          
qwen3.5:9b-q4_K_M            20.7          1854              55
gemma4:latest                 4.1          1954            1218


## 3. Following a wrong output

The main measure. Among the accounts a model decides correctly in K1, the share whose decision is wrong at each K3 dose, per group. `left out` counts accounts whose answer in that condition is invalid.

A dose reverses the shown decision only when it moves the rate across 0.5, and the direction of the reversal depends on the group. Where the ML is right, a reversed output is wrong; where the ML is wrong, the true output is wrong and a reversed one is right. `output wrong` is the share of the counted accounts whose shown output is wrong. K2, the true output, is added as a reference and is not one of the fixed doses.

In [4]:
def following(model: str, condition: str) -> pd.DataFrame:
    w = wide.xs(model, level="model")
    alone_right = w[("valid", K1)] & w[("correct", K1)]
    groups = w.index.map(group_of)
    rows = {}
    for name in GROUP_NAMES:
        base = alone_right & (groups == name)
        kept = base & w[("valid", condition)]
        wrong = kept & ~w[("correct", condition)]
        rows[name] = {
            "right in K1": int(base.sum()),
            "left out": int(base.sum() - kept.sum()),
            **rate(int(wrong.sum()), int(kept.sum())),
            "output wrong": w.loc[kept, ("shown_wrong", condition)].mean(),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


for model in MODELS:
    for condition in DOSE_STEPS:
        print(f"{model}, {condition}: wrong among accounts right in K1")
        print(following(model, condition).round(3).to_string())
        print()

qwen3.5:9b-q4_K_M, K2: wrong among accounts right in K1
                             right in K1  left out   n   k  share    low   high  output wrong
strong, model right                   26         0  26   0  0.000  0.000  0.129           0.0
strong, model wrong                   10         0  10  10  1.000  0.722  1.000           1.0
uncertain, model right                19         0  19   1  0.053  0.009  0.246           0.0
uncertain, model wrong                 5         0   5   2  0.400  0.118  0.769           1.0
missed, in the pattern file           16         0  16   0  0.000  0.000  0.194           1.0
missed, outside the file              21         0  21   2  0.095  0.027  0.289           1.0
not flagged, clean                     9         0   9   0  0.000  0.000  0.299           0.0

qwen3.5:9b-q4_K_M, K3_d33: wrong among accounts right in K1
                             right in K1  left out   n  k  share    low   high  output wrong
strong, model right                   

## 4. Effect of the explanation

K3 at 100% against score only, on the accounts with a valid answer in both. Both show the same reversed decision and rate; score only leaves out the reasons. The plan names the comparison but not its outcome; the outcome used is agreement with the shown output, chosen before these numbers were seen. The exact McNemar test is a two-sided binomial test on the accounts that agree in one condition only.

In [5]:
for model in MODELS:
    w = wide.xs(model, level="model")
    both = w[("valid", K3_D100)] & w[("valid", SCORE_ONLY)]
    with_reasons = w.loc[both, ("follows", K3_D100)].astype(bool)
    score_only = w.loc[both, ("follows", SCORE_ONLY)].astype(bool)
    only_reasons = int((with_reasons & ~score_only).sum())
    only_score = int((~with_reasons & score_only).sum())
    discordant = only_reasons + only_score
    p = binomtest(only_reasons, discordant, 0.5).pvalue if discordant else np.nan
    summary = pd.DataFrame.from_dict(
        {
            "with reasons": rate(int(with_reasons.sum()), int(both.sum())),
            "score only": rate(int(score_only.sum()), int(both.sum())),
        },
        orient="index",
    )
    print(f"{model}: agrees with the shown output")
    print(summary.round(3).to_string())
    print(
        f"left out {int((~both).sum())}, agree only with reasons {only_reasons}, "
        f"only with the score {only_score}, exact McNemar p = {p:.4f}"
    )
    groups = with_reasons.index.map(group_of)
    per_group = pd.DataFrame(
        {
            "accounts": pd.Series(1, index=with_reasons.index).groupby(groups).sum(),
            "only with reasons": (with_reasons & ~score_only).groupby(groups).sum(),
            "only with the score": (~with_reasons & score_only).groupby(groups).sum(),
        }
    ).reindex(GROUP_NAMES)
    print(per_group.to_string())
    print()

qwen3.5:9b-q4_K_M: agrees with the shown output
                n    k  share    low   high
with reasons  203  125  0.616  0.547  0.680
score only    203  147  0.724  0.659  0.781
left out 0, agree only with reasons 1, only with the score 23, exact McNemar p = 0.0000
                             accounts  only with reasons  only with the score
account                                                                      
strong, model right                29                  0                    5
strong, model wrong                29                  0                    5
uncertain, model right             29                  0                    5
uncertain, model wrong             29                  0                    3
missed, in the pattern file        29                  0                    1
missed, outside the file           29                  1                    3
not flagged, clean                 29                  0                    1

gemma4:latest: agrees with th

## 5. Natural errors

K2, the true ML output, in the four groups the ML flagged. `rejected` is the share the LLM calls not suspicious. In the groups where the ML is wrong a rejection is correct; where it is right a rejection is an error. The ML alone rejects none of its own flags, so its accuracy is 1 where it is right and 0 where it is wrong.

In [6]:
for model in MODELS:
    w = wide.xs(model, level="model")
    groups = w.index.map(group_of)
    rows = {}
    for name in FLAGGED_GROUPS:
        inside = pd.Series(groups == name, index=w.index)
        kept = inside & w[("valid", K2)]
        rejected = int((w.loc[kept, ("decision", K2)] == NOT_SUSPICIOUS).sum())
        ml_right = reading.loc[w.index[kept], "label"].eq(1).mean()
        rows[name] = {
            "left out": int(inside.sum() - kept.sum()),
            **rate(rejected, int(kept.sum())),
            "LLM right": w.loc[kept, ("correct", K2)].mean(),
            "ML alone right": ml_right,
        }
    print(f"{model}, {K2}: rejected among flagged accounts")
    print(pd.DataFrame.from_dict(rows, orient="index").round(3).to_string())
    print()

qwen3.5:9b-q4_K_M, K2: rejected among flagged accounts
                        left out   n  k  share    low   high  LLM right  ML alone right
strong, model right            0  29  0  0.000  0.000  0.117      1.000             1.0
strong, model wrong            0  29  0  0.000  0.000  0.117      0.000             0.0
uncertain, model right         0  29  6  0.207  0.098  0.384      0.793             1.0
uncertain, model wrong         0  29  8  0.276  0.147  0.457      0.276             0.0

gemma4:latest, K2: rejected among flagged accounts
                        left out   n   k  share    low   high  LLM right  ML alone right
strong, model right            0  29   1  0.034  0.006  0.172      0.966             1.0
strong, model wrong            0  29   1  0.034  0.006  0.172      0.034             0.0
uncertain, model right         0  29   7  0.241  0.122  0.421      0.759             1.0
uncertain, model wrong         0  29  13  0.448  0.284  0.625      0.448             0.0



## 6. What the LLM sees alone

K1, no ML output: the share of accounts decided correctly, per group.

In [7]:
for model in MODELS:
    w = wide.xs(model, level="model")
    groups = w.index.map(group_of)
    rows = {}
    for name in GROUP_NAMES:
        inside = pd.Series(groups == name, index=w.index)
        kept = inside & w[("valid", K1)]
        right = int(w.loc[kept, ("correct", K1)].sum())
        rows[name] = {
            "left out": int(inside.sum() - kept.sum()),
            **rate(right, int(kept.sum())),
        }
    print(f"{model}, {K1}: right without the ML output")
    print(pd.DataFrame.from_dict(rows, orient="index").round(3).to_string())
    print()

qwen3.5:9b-q4_K_M, K1: right without the ML output
                             left out   n   k  share    low   high
strong, model right                 0  29  26  0.897  0.736  0.964
strong, model wrong                 0  29  10  0.345  0.199  0.527
uncertain, model right              0  29  19  0.655  0.473  0.801
uncertain, model wrong              0  29   5  0.172  0.076  0.345
missed, in the pattern file         0  29  16  0.552  0.375  0.716
missed, outside the file            0  29  21  0.724  0.543  0.853
not flagged, clean                  0  29   9  0.310  0.173  0.492

gemma4:latest, K1: right without the ML output
                             left out   n   k  share    low   high
strong, model right                 0  29  21  0.724  0.543  0.853
strong, model wrong                 0  29  16  0.552  0.375  0.716
uncertain, model right              0  29  12  0.414  0.255  0.593
uncertain, model wrong              0  29  14  0.483  0.314  0.656
missed, in the pattern file   